# Bouaddi vs L. Camara vs M. Sangaré — Three-Way Midfielder Comparison
## WhoScored / Opta Event Data · 2025/26 Ligue 1

**Layout:** 2-top / 1-bottom pitch format for all maps

**Outputs — 12 visuals:**
1. `01_touch_map.png`
2. `02_pass_map.png` — thin arrows (normal), thick arrows (final third), star-ended (into box)
3. `03_fast_pass_map.png` — passes within 2 s of receiving the ball
4. `04_progressive_carries.png` — ≥10 Opta units forward; box entries labelled
5. `05_post_carry_action.png` — what happens after a progressive carry (arrow=pass, dot=shot)
6. `06_ball_recovery_map.png` — tackles ★, interceptions ▲, recoveries ●
7. `07_post_recovery_pass_map.png` — passes within 3 s of ball recovery
8. `08_xt_zone_comparison.png` — three-player xT via passes in one visual (3 mini-pitches)
9. `09_pre_receive_distance.png` — avg distance covered before receiving
10. `10_pre_defensive_distance.png` — avg distance covered before a defensive action

---
**Coordinate system:** Opta — `x`: 0→100 (own→opp goal), `y`: 0→100 (bottom→top).  
`pitch_type='opta'` in mplsoccer maps these natively — **no flipping needed**.


In [ ]:
# Uncomment to install dependencies if needed
#!pip install mplsoccer matplotlib pandas numpy selenium webdriver-manager requests beautifulsoup4 lxml scipy -q

In [1]:
"""
================================================================
  CONFIGURATION  —  edit match_suffixes before running
================================================================
  WhoScored player fixture pages (2025/26 Ligue 1):
    Bouaddi : https://www.whoscored.com/players/504830/fixtures/ayyoub-bouaddi
    L. Camara: https://www.whoscored.com/players/447689/fixtures/lamine-camara
    M. Sangaré: https://www.whoscored.com/players/442971/fixtures/mamadou-sangare

  HOW TO FIND MATCH IDs:
    1. Visit any WhoScored match page, e.g.:
       https://www.whoscored.com/Matches/1906XXX/Live
    2. The full numeric ID is in the URL after /Matches/
    3. All 2025/26 Ligue 1 matches share the same prefix (e.g. '1906').
       Extract the prefix and suffixes as shown in the config below.
    4. From each player's /fixtures/ page, hover a match link to see its ID in the status bar.

  EXAMPLE:
    Match URL: https://www.whoscored.com/Matches/1906142/Live
    → match_prefix  = '1906'
    → match_suffixes = [142, ...]

  Replace the placeholder suffixes below with the real ones from each
  player's fixtures page for 2025/26 Ligue 1 (midfield starts only).
"""

import json, time, random, warnings, re, math
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.cm import ScalarMappable
from mplsoccer import Pitch

warnings.filterwarnings('ignore')

# ── Visual palette ─────────────────────────────────────────────
BG_COLOR    = "#FFF8E7"   # light amber
TEXT_COLOR  = "#1A1A1A"
PITCH_LINE  = "#C4A882"   # warm tan
GOLD        = "#F6A623"
GREY        = "#AAAAAA"
CREDIT_LINE = "Data: Opta | xTransfers Episode 34"

# ── Competition ────────────────────────────────────────────────
COMPETITION = "2025/26 Ligue 1"

# ── Players ────────────────────────────────────────────────────
# Each entry: name, player_id, team, color, match_prefix, match_suffixes
#
# ★ FILL IN match_suffixes from the WhoScored fixture pages ★
#
PLAYERS = [
    {
        "name"           : "A. Bouaddi",
        "player_id"      : 504830,
        "team"           : "Lille",
        "color"          : "#C62828",   # deep red  (Lille away)
        "match_prefix"   : "1911",      # ← Ligue 1 25/26 prefix (verify first match URL)
        "match_suffixes" : [281, 279, 310, 336, 360, 374, 307, 335,
                            308, 337, 353, 383, 392, 405, 419, 431, 438,
                            447, 458, 471, 492, 505, 510, 530, 547, 560, 564, 577
        ],
    },
    {
        "name"           : "L. Camara",
        "player_id"      : 447689,
        "team"           : "AS Monaco",
        "color"          : "#D32F2F",   # Monaco red
        "match_prefix"   : "1911",
        "match_suffixes" : [            # ← PLACEHOLDER — update before running
            290, 279, 318, 330, 366, 400, 415, 382, 406, 437, 448, 460, 468, 474, 485, 496, 503,
            509, 533, 542, 562, 566, 577, 528
        ],
    },
    {
        "name"           : "M. Sangaré",
        "player_id"      : 442971,
        "team"           : "RC Lens",
        "color"          : "#1565C0",   # PSG blue
        "match_prefix"   : "1911",
        "match_suffixes" : [            # ← PLACEHOLDER — update before running
            276, 295, 348, 360, 294, 301, 326, 306, 331, 351, 379, 390, 403, 389, 402, 427, 439, 445, 
            457, 469, 488, 491, 500, 508, 530, 546, 555, 568, 538
        ],
    },
]

# Build full integer match-ID lists from prefix + suffix
for _p in PLAYERS:
    if _p['match_prefix'] == "":
        _p["match_ids"] = [int(s) for s in _p['match_suffixes']]
    else:
        _p["match_ids"] = [
            int(f"{_p['match_prefix']}{str(s).zfill(3)}")
            for s in _p["match_suffixes"]
        ]

# ── Output dir ─────────────────────────────────────────────────
OUTPUT_DIR = Path("./bouaddi_camara_sangare")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Players configured:")
for _p in PLAYERS:
    example_id = _p['match_ids'][0] if _p['match_ids'] else 'none'
    print(f"  {_p['name']:<16} id={_p['player_id']}  matches={len(_p['match_ids'])}  "
          f"example_match_id={example_id}")
print("\n⚠  Remember to replace placeholder match_suffixes with real IDs before running!")

Players configured:
  A. Bouaddi       id=504830  matches=28  example_match_id=1911281
  L. Camara        id=447689  matches=24  example_match_id=1911290
  M. Sangaré       id=442971  matches=29  example_match_id=1911276

⚠  Remember to replace placeholder match_suffixes with real IDs before running!


In [2]:
# ═══════════════════════════════════════════════════════════════
# OPTA EVENT TYPES + LOW-LEVEL HELPERS
# ═══════════════════════════════════════════════════════════════
#
#  1  Pass          3  TakeOn         7  Tackle
#  8  Interception  49 BallRecovery   67 Pressure
# 10  SavedShot    11  ShotOffTarget  12  ShotBlocked
# 13  Miss         14  Post           16  Goal
# 50  Dispossessed 51  Error

SHOT_TYPES            = {10, 11, 12, 13, 14, 16}
POSSESSION_EVENTS     = {1, 3, 10, 11, 12, 13, 14, 16}
DEFENSIVE_ACTION_TYPES = {7, 8, 49}   # Tackle, Interception, BallRecovery
QUALIFIER_OVERRUN     = 211
QUALIFIER_FREEKICK    = 5    # Free kick taken (direct or indirect)
QUALIFIER_CORNER      = 6    # Corner taken
SET_PIECE_QUALIFIERS  = {QUALIFIER_FREEKICK, QUALIFIER_CORNER}


def get_type_value(ev):
    t = ev.get('type', {})
    return t.get('value', -1) if isinstance(t, dict) else (int(t) if t else -1)


def is_successful(ev):
    oc = ev.get('outcomeType', {})
    return (oc.get('value', 0) if isinstance(oc, dict) else 0) == 1


def has_qualifier(ev, qid):
    return any(q.get('type', {}).get('value') == qid for q in ev.get('qualifiers', []))


def event_time(ev):
    """Absolute match time in seconds."""
    return ev.get('minute', 0) * 60.0 + ev.get('second', 0.0)


def xy(ev):        return ev.get('x'),    ev.get('y')
def end_xy(ev):    return ev.get('endX'), ev.get('endY')
def _safe_xs(evs): return [e['x'] for e in evs if e.get('x') is not None]
def _safe_ys(evs): return [e['y'] for e in evs if e.get('y') is not None]

print("Event helpers ready.")

Event helpers ready.


In [3]:
# ═══════════════════════════════════════════════════════════════
# EVENT FILTER FUNCTIONS
# ═══════════════════════════════════════════════════════════════

def is_set_piece_pass(ev):
    """True if this pass event is a corner or a free kick (direct/indirect)."""
    return any(has_qualifier(ev, q) for q in SET_PIECE_QUALIFIERS)


def filter_passes(player_events, open_play_only=True):
    """
    Returns (succ_passes, fail_passes).
    open_play_only=True (default) excludes corners and free kicks (qualifiers
    6 and 5), so the pass map / xT-per-zone reflect open-play distribution only.
    """
    passes = [ev for ev in player_events if get_type_value(ev) == 1]
    if open_play_only:
        passes = [ev for ev in passes if not is_set_piece_pass(ev)]
    return (
        [ev for ev in passes if is_successful(ev)],
        [ev for ev in passes if not is_successful(ev)],
    )


def filter_final_third_passes(succ_passes):
    """
    Returns (final_third, box_passes).
    Final third: endX > 66.67
    Box passes : endX > 83.0 AND endY ∈ [21.1, 78.9]  (Opta penalty-area bounds)
    """
    final_third, box_passes = [], []
    for ev in succ_passes:
        ex, ey = end_xy(ev)
        if ex is None:
            continue
        if ex > 66.67:
            final_third.append(ev)
            if ex > 83.0 and 21.1 <= (ey or 0) <= 78.9:
                box_passes.append(ev)
    return final_third, box_passes


def filter_tackles_interceptions(player_events):
    tack_won  = [ev for ev in player_events if get_type_value(ev) == 7 and is_successful(ev)]
    tack_lost = [ev for ev in player_events if get_type_value(ev) == 7 and not is_successful(ev)]
    intercept = [ev for ev in player_events if get_type_value(ev) == 8]
    return tack_won, tack_lost, intercept


def filter_ground_duels(player_events):
    tackles = [ev for ev in player_events if get_type_value(ev) == 7]
    takeons = [
        ev for ev in player_events
        if get_type_value(ev) == 3 and not has_qualifier(ev, QUALIFIER_OVERRUN)
    ]
    all_duels = tackles + takeons
    return (
        [ev for ev in all_duels if is_successful(ev)],
        [ev for ev in all_duels if not is_successful(ev)],
    )


def filter_fast_passes(all_events, player_id, max_seconds=2.0):
    """
    Passes released within max_seconds (2 s) of the preceding event.
    Returns (fast_succ, fast_fail).
    """
    sorted_evs = sorted(all_events, key=event_time)
    fast_succ, fast_fail = [], []
    for i, ev in enumerate(sorted_evs):
        if ev.get('playerId') != player_id or get_type_value(ev) != 1:
            continue
        if i == 0:
            continue
        # Find the most recent PRIOR event by ANY player (ball played to this player)
        dt = event_time(ev) - event_time(sorted_evs[i - 1])
        if 0 <= dt <= max_seconds:
            (fast_succ if is_successful(ev) else fast_fail).append(ev)
    return fast_succ, fast_fail


def filter_post_recovery_passes(all_events, player_id, max_seconds=3.0):
    """
    Passes made within max_seconds of a ball recovery action by the player.
    Recovery actions: tackle won (7), interception (8), ball recovery (49).
    Each defensive action credits at most one subsequent pass.
    Returns (post_rec_succ, post_rec_fail).
    """
    sorted_evs = sorted(all_events, key=event_time)
    player_evs = [ev for ev in sorted_evs if ev.get('playerId') == player_id]

    def_times = []
    for ev in player_evs:
        t = get_type_value(ev)
        if t in DEFENSIVE_ACTION_TYPES:
            if t == 7 and not is_successful(ev):
                continue
            def_times.append(event_time(ev))
    def_times_sorted = sorted(def_times)

    post_rec_succ, post_rec_fail = [], []
    used = set()

    for ev in player_evs:
        if get_type_value(ev) != 1:
            continue
        t_pass = event_time(ev)
        for dt_def in reversed(def_times_sorted):
            if dt_def > t_pass:
                continue
            gap = t_pass - dt_def
            if gap > max_seconds:
                break
            if dt_def in used:
                continue
            used.add(dt_def)
            (post_rec_succ if is_successful(ev) else post_rec_fail).append(ev)
            break

    return post_rec_succ, post_rec_fail


def reconstruct_carries(all_events, player_id):
    """
    Infer carries from gaps between consecutive possession events.
    Criteria: displacement 2.5–35 Opta units, time gap ≤ 12 s.
    Returns list of carry dicts {x, y, endX, endY, distance}.
    """
    player_evs = [
        ev for ev in all_events
        if ev.get('playerId') == player_id
        and get_type_value(ev) in POSSESSION_EVENTS
    ]
    player_evs.sort(key=event_time)
    carries = []
    for i in range(len(player_evs) - 1):
        curr, nxt = player_evs[i], player_evs[i + 1]
        ex, ey = end_xy(curr)
        nx, ny = xy(nxt)
        if None in (ex, ey, nx, ny):
            continue
        dt   = event_time(nxt) - event_time(curr)
        if dt < 0 or dt > 12:
            continue
        dist = ((ex - nx) ** 2 + (ey - ny) ** 2) ** 0.5
        if dist < 2.5 or dist > 35.0:
            continue
        carries.append({'x': ex, 'y': ey, 'endX': nx, 'endY': ny,
                        'distance': dist, 'time': event_time(curr),
                        'match_id': curr.get('match_id')})
    return carries


def is_progressive_carry(c):  return (c['endX'] - c['x']) >= 10.0
def is_carry_into_box(c):     return c['x'] < 83.0 and c['endX'] >= 83.0 and 21.1 <= c['endY'] <= 78.9


def get_post_carry_action(all_events, player_id, carries):
    """
    For each progressive carry, find the very next event by this player.
    Returns list of dicts: {carry, next_event_type, sx, sy, ex, ey}
    """
    sorted_evs = sorted(all_events, key=event_time)
    player_evs = [ev for ev in sorted_evs if ev.get('playerId') == player_id]

    results = []
    for carry in carries:
        if not is_progressive_carry(carry):
            continue
        t_carry_end = carry['time']  # approx end time of carry
        # Find next player event after the carry's end position time
        for ev in player_evs:
            t_ev = event_time(ev)
            if t_ev <= t_carry_end:
                continue
            if t_ev - t_carry_end > 12:  # max 12 s gap
                break
            tv = get_type_value(ev)
            sx, sy = carry['endX'], carry['endY']
            ex, ey = end_xy(ev) if tv == 1 else (None, None)
            results.append({
                'carry'           : carry,
                'next_type'       : tv,
                'is_pass'         : tv == 1,
                'is_shot'         : tv in SHOT_TYPES,
                'is_successful'   : is_successful(ev),
                'sx'              : sx,  'sy'  : sy,
                'ex'              : ex,  'ey'  : ey,
                'ex_shot'         : ev.get('x') if tv in SHOT_TYPES else None,
                'ey_shot'         : ev.get('y') if tv in SHOT_TYPES else None,
            })
            break
    return results


def compute_pre_receive_distances(all_events, player_id):
    """
    Estimate distance covered before receiving the ball.
    Proxy: for each 'receive' (first touch after a pass by another player),
    find the previous event by this player and compute Euclidean displacement.
    A player's own last position → current reception point.
    Returns list of distances (Opta units).
    """
    sorted_evs = sorted(all_events, key=event_time)
    distances  = []
    last_pos   = {}  # player_id → (x, y, time)

    for ev in sorted_evs:
        pid = ev.get('playerId')
        tv  = get_type_value(ev)
        px, py = xy(ev)
        if None in (px, py):
            continue

        if pid == player_id:
            # If this player appears after a gap, estimate movement
            if pid in last_pos:
                lx, ly, lt = last_pos[pid]
                dt = event_time(ev) - lt
                if 0 < dt <= 10:  # reasonable gap
                    dist = math.sqrt((px - lx)**2 + (py - ly)**2)
                    if dist > 0.5:  # minimum meaningful movement
                        distances.append(dist)
            last_pos[pid] = (px, py, event_time(ev))
        else:
            # Another player's event — update their last position
            ex_, ey_ = end_xy(ev)
            if tv == 1 and ex_ is not None:  # pass
                last_pos[pid] = (ex_, ey_, event_time(ev))
            else:
                last_pos[pid] = (px, py, event_time(ev))

    return distances


def compute_pre_defensive_distances(all_events, player_id):
    """
    Distance covered before making a defensive action.
    Proxy: displacement from the player's previous event location to
    the defensive action location.
    Returns list of distances (Opta units).
    """
    sorted_evs  = sorted(all_events, key=event_time)
    player_evs  = [ev for ev in sorted_evs if ev.get('playerId') == player_id]
    distances   = []

    for i, ev in enumerate(player_evs):
        tv = get_type_value(ev)
        if tv not in DEFENSIVE_ACTION_TYPES:
            continue
        px, py = xy(ev)
        if None in (px, py):
            continue
        if i == 0:
            continue
        prev = player_evs[i - 1]
        ppx, ppy = xy(prev)
        if None in (ppx, ppy):
            continue
        dt = event_time(ev) - event_time(prev)
        if 0 < dt <= 15:
            dist = math.sqrt((px - ppx)**2 + (py - ppy)**2)
            if dist > 0.3:
                distances.append(dist)

    return distances


print("Event filter functions ready.")

Event filter functions ready.


In [4]:
# ═══════════════════════════════════════════════════════════════
# EXPECTED THREAT (xT) — Karun Singh 16×12 grid
# ═══════════════════════════════════════════════════════════════

XT_GRID = np.array([
    [0.000,0.000,0.001,0.001,0.001,0.002,0.002,0.003,0.004,0.006,0.008,0.013,0.022,0.042,0.068,0.112],
    [0.000,0.000,0.001,0.001,0.002,0.002,0.003,0.004,0.005,0.008,0.012,0.019,0.032,0.062,0.094,0.143],
    [0.000,0.001,0.001,0.002,0.002,0.003,0.004,0.005,0.007,0.011,0.016,0.024,0.040,0.072,0.113,0.168],
    [0.000,0.001,0.001,0.002,0.003,0.003,0.004,0.006,0.008,0.013,0.019,0.028,0.046,0.082,0.128,0.184],
    [0.000,0.001,0.001,0.002,0.003,0.004,0.005,0.007,0.009,0.014,0.021,0.031,0.052,0.090,0.138,0.198],
    [0.000,0.001,0.001,0.002,0.003,0.004,0.006,0.008,0.011,0.016,0.023,0.035,0.057,0.098,0.153,0.224],
    [0.000,0.001,0.001,0.002,0.003,0.004,0.006,0.008,0.011,0.016,0.023,0.035,0.057,0.098,0.153,0.224],
    [0.000,0.001,0.001,0.002,0.003,0.004,0.005,0.007,0.009,0.014,0.021,0.031,0.052,0.090,0.138,0.198],
    [0.000,0.001,0.001,0.002,0.003,0.003,0.004,0.006,0.008,0.013,0.019,0.028,0.046,0.082,0.128,0.184],
    [0.000,0.001,0.001,0.002,0.002,0.003,0.004,0.005,0.007,0.011,0.016,0.024,0.040,0.072,0.113,0.168],
    [0.000,0.000,0.001,0.001,0.002,0.002,0.003,0.004,0.005,0.008,0.012,0.019,0.032,0.062,0.094,0.143],
    [0.000,0.000,0.001,0.001,0.001,0.002,0.002,0.003,0.004,0.006,0.008,0.013,0.022,0.042,0.068,0.112],
])  # shape: (12 rows y-axis, 16 cols x-axis)

N_ROWS, N_COLS = XT_GRID.shape   # 12, 16
CELL_W = 100 / N_COLS            # 6.25 Opta x units per cell
CELL_H = 100 / N_ROWS            # 8.33 Opta y units per cell


def get_xt_zone(x, y):
    """Return (col, row) indices for Opta coordinate (x, y)."""
    col = min(int(x / 100 * N_COLS), N_COLS - 1)
    row = min(int(y / 100 * N_ROWS), N_ROWS - 1)
    return col, row


def get_xt_value(x, y):
    col, row = get_xt_zone(x, y)
    return float(XT_GRID[row, col])


def compute_pass_xt_full(succ_passes):
    """Compute xT start, end, gained for every successful pass."""
    results = []
    for ev in succ_passes:
        sx, sy = xy(ev)
        ex, ey = end_xy(ev)
        if None in (sx, sy, ex, ey):
            continue
        xt_s = get_xt_value(sx, sy)
        xt_e = get_xt_value(ex, ey)
        results.append({
            'x': sx, 'y': sy, 'endX': ex, 'endY': ey,
            'xt_start' : xt_s,
            'xt_end'   : xt_e,
            'xt_gained': xt_e - xt_s,
        })
    return results


def compute_xt_zones_net(pass_xt_data):
    """Aggregate net xT per origin zone."""
    zone_net = np.zeros((N_ROWS, N_COLS))
    for d in pass_xt_data:
        col, row = get_xt_zone(d['x'], d['y'])
        zone_net[row, col] += d['xt_gained']
    return zone_net


print("xT helpers ready.")

xT helpers ready.


In [5]:
# ═══════════════════════════════════════════════════════════════
# SELENIUM DRIVER + WHOSCORED SCRAPER
# ═══════════════════════════════════════════════════════════════

def get_driver():
    from selenium import webdriver
    from selenium.webdriver.chrome.options import Options
    from selenium.webdriver.chrome.service import Service
    from webdriver_manager.chrome import ChromeDriverManager

    opts = Options()
    opts.add_argument('--headless=new')
    opts.add_argument('--no-sandbox')
    opts.add_argument('--disable-dev-shm-usage')
    opts.add_argument('--window-size=1920,1080')
    opts.add_argument('--disable-blink-features=AutomationControlled')
    opts.add_experimental_option('excludeSwitches', ['enable-automation'])
    opts.add_argument(
        'user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36'
    )
    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()), options=opts
    )
    driver.execute_script(
        "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
    )
    return driver


def scrape_match(driver, match_id, player_id, player_name):
    """
    Fetch all Opta events for a WhoScored match.
    Returns (all_events, player_team_id) or ([], None) on failure.
    """
    import json as _json

    url = f'https://www.whoscored.com/Matches/{match_id}/Live'
    print(f'  [WS] Fetching match {match_id} ...')
    driver.get(url)
    time.sleep(random.uniform(10, 16))
    src = driver.page_source

    marker_pos = src.find('matchCentreData')
    if marker_pos == -1:
        print(f'       matchCentreData not found — skipping {match_id}')
        return [], None

    json_start = src.find('{', marker_pos)
    if json_start == -1:
        print(f'       No opening brace — skipping {match_id}')
        return [], None

    depth = 0; in_string = False; escape = False; end_idx = json_start
    for i, ch in enumerate(src[json_start:], json_start):
        if escape:                      escape = False; continue
        if ch == '\\' and in_string:   escape = True;  continue
        if ch == '"' and not escape:   in_string = not in_string; continue
        if in_string:                  continue
        if ch == '{':                  depth += 1
        elif ch == '}':                depth -= 1
        if depth == 0:                 end_idx = i; break

    try:
        data = _json.loads(src[json_start: end_idx + 1])
    except Exception as e:
        print(f'       Parse error {match_id}: {e}')
        return [], None

    events = data.get('events', [])
    if not events:
        return [], None

    found_name, player_team_id = None, None
    for side in ('home', 'away'):
        side_data = data.get(side, {})
        for p in side_data.get('players', []):
            pid = p.get('playerId') or p.get('id')
            if pid == player_id:
                found_name     = p.get('name', player_name)
                player_team_id = side_data.get('teamId')
                break
        if found_name:
            break

    if not found_name:
        print(f'       {player_name} not in squad — skipping {match_id}')
        return [], None

    for ev in events:
        ev['match_id'] = match_id

    n = sum(1 for ev in events if ev.get('playerId') == player_id)
    print(f'       ✓ {found_name} — {n} raw events')
    return events, player_team_id


def process_match(all_events, player_id):
    """Extract and categorise all events for one player from a single match."""
    player_evs = [ev for ev in all_events if ev.get('playerId') == player_id]

    succ_passes, fail_passes       = filter_passes(player_evs)
    final_third, box_passes        = filter_final_third_passes(succ_passes)
    tack_won, tack_lost, intercept = filter_tackles_interceptions(player_evs)
    loose_rec  = [ev for ev in player_evs if get_type_value(ev) == 49]
    carries    = reconstruct_carries(all_events, player_id)
    prog_carries  = [c for c in carries if is_progressive_carry(c)]
    box_carries   = [c for c in carries if is_carry_into_box(c)]
    fast_succ, fast_fail = filter_fast_passes(all_events, player_id, max_seconds=2.0)
    post_rec_succ, post_rec_fail = filter_post_recovery_passes(all_events, player_id, max_seconds=3.0)
    shots   = [ev for ev in player_evs if get_type_value(ev) in SHOT_TYPES]
    post_carry_actions = get_post_carry_action(all_events, player_id, prog_carries)
    pre_recv_dists = compute_pre_receive_distances(all_events, player_id)
    pre_def_dists  = compute_pre_defensive_distances(all_events, player_id)

    return {
        'succ_passes'       : succ_passes,
        'fail_passes'       : fail_passes,
        'final_third_passes': final_third,
        'box_passes'        : box_passes,
        'tack_won'          : tack_won,
        'tack_lost'         : tack_lost,
        'interceptions'     : intercept,
        'loose_rec'         : loose_rec,
        'carries'           : carries,
        'prog_carries'      : prog_carries,
        'box_carries'       : box_carries,
        'fast_succ'         : fast_succ,
        'fast_fail'         : fast_fail,
        'post_rec_succ'     : post_rec_succ,
        'post_rec_fail'     : post_rec_fail,
        'shots'             : shots,
        'post_carry_actions': post_carry_actions,
        'pre_recv_dists'    : pre_recv_dists,
        'pre_def_dists'     : pre_def_dists,
    }


def scrape_player(driver, player_cfg):
    """Scrape all matches for one player. Returns (accumulated_dict, game_count)."""
    acc = {
        'succ_passes':[], 'fail_passes':[], 'final_third_passes':[], 'box_passes':[],
        'tack_won':[], 'tack_lost':[], 'interceptions':[], 'loose_rec':[],
        'carries':[], 'prog_carries':[], 'box_carries':[],
        'fast_succ':[], 'fast_fail':[],
        'post_rec_succ':[], 'post_rec_fail':[],
        'shots':[], 'post_carry_actions':[],
        'pre_recv_dists':[], 'pre_def_dists':[],
    }
    games = 0
    for mid in player_cfg['match_ids']:
        all_evs, _ = scrape_match(driver, mid, player_cfg['player_id'], player_cfg['name'])
        if not all_evs:
            continue
        m = process_match(all_evs, player_cfg['player_id'])
        for k in acc:
            acc[k].extend(m.get(k, []))
        games += 1
        time.sleep(random.uniform(10, 18))
    print(f'  → {player_cfg["name"]}: {games} games successfully scraped')
    return acc, games


print("Scraper ready.")

Scraper ready.


In [6]:
# ═══════════════════════════════════════════════════════════════
# PITCH + LAYOUT HELPERS  (3-way: 2 top / 1 bottom)
# ═══════════════════════════════════════════════════════════════

def _pitch():
    return Pitch(
        pitch_type  = 'opta',
        pitch_color = BG_COLOR,
        line_color  = PITCH_LINE,
        linewidth   = 1.4,
        goal_type   = 'box',
        corner_arcs = True,
    )


def _make_3way_fig(title, subtitle, note=''):
    """
    Creates a 2-top / 1-bottom layout.
    Row 0: ax0 (left)  ax1 (right)   ← players 0 and 1
    Row 1: ax2 (centre, spans full)  ← player 2
    Returns (fig, [ax0, ax1, ax2]).
    """
    fig = plt.figure(figsize=(34, 28))
    fig.set_facecolor(BG_COLOR)

    # GridSpec: 2 rows, 2 cols; bottom row's centre is spanned by 1 axis
    from matplotlib.gridspec import GridSpec
    gs = GridSpec(
        2, 2,
        figure   = fig,
        hspace   = 0.18,
        wspace   = 0.06,
        top      = 0.88,
        bottom   = 0.02,
        left     = 0.01,
        right    = 0.99,
    )
    ax0 = fig.add_subplot(gs[0, 0])  # top-left
    ax1 = fig.add_subplot(gs[0, 1])  # top-right
    ax2 = fig.add_subplot(gs[1, :])  # bottom, full width

    p = _pitch()
    for ax in [ax0, ax1, ax2]:
        p.draw(ax=ax)
        ax.set_facecolor(BG_COLOR)

    # Titles
    fig.text(0.5, 1.07, title,
             ha='center', va='top', fontsize=45, fontweight='bold', color=TEXT_COLOR)
    fig.text(0.5, 1.02, subtitle,
             ha='center', va='top', fontsize=32, color=TEXT_COLOR, alpha=0.95)
    fig.text(0.5, 0.97, CREDIT_LINE,
             ha='center', va='top', fontsize=25, color=TEXT_COLOR, alpha=0.95, style='italic')
    if note:
        fig.text(0.5, 0.93, note,
                 ha='center', va='top', fontsize=19, color=TEXT_COLOR, alpha=0.9, style='italic')

    return fig, [ax0, ax1, ax2]


def _label(ax, player_cfg):
    ax.set_title(
        f"{player_cfg['name']}  ·  {player_cfg['team']}",
        fontsize=32, fontweight='bold', color=player_cfg['color'], pad=9,
    )


def _stats_box(ax, txt, color=None):
    if color is None:
        color = TEXT_COLOR
    ax.text(0.98, 0.04, txt, transform=ax.transAxes, ha='right', va='bottom',
            fontsize=16, color=TEXT_COLOR,
            bbox=dict(boxstyle='round,pad=0.4', facecolor=BG_COLOR,
                      edgecolor=color, alpha=0.99),
            fontfamily='monospace')


print("Layout helpers ready.")

Layout helpers ready.


In [7]:
# ═══════════════════════════════════════════════════════════════
# PLOT 1 — TOUCH MAP
# ═══════════════════════════════════════════════════════════════

def plot_touch_map(all_accs, all_n):
    p = _pitch()
    fig, axes = _make_3way_fig(
        title    = f'{PLAYERS[0]["name"]}  vs  {PLAYERS[1]["name"]}  vs  {PLAYERS[2]["name"]}  —  Touch Map',
        subtitle = f'{COMPETITION}  ·  KDE heatmap of all touches  ·  attacking left → right',
    )

    for ax, player_cfg, acc, n in zip(axes, PLAYERS, all_accs, all_n):
        _label(ax, player_cfg)
        color = player_cfg['color']
        pts   = acc['succ_passes'] + acc['fail_passes'] + acc['tack_won'] + acc['tack_lost'] + \
                acc['interceptions'] + acc['loose_rec'] + acc['shots']
        xs = _safe_xs(pts)
        ys = _safe_ys(pts)
        if len(xs) >= 5:
            cmap = LinearSegmentedColormap.from_list('touch', [BG_COLOR, '#FDD835', color], N=256)
            p.kdeplot(xs, ys, ax=ax, fill=True, levels=60,
                      cmap=cmap, alpha=0.80, zorder=2, bw_adjust=0.9)
        p.scatter(xs, ys, ax=ax, s=12, color=color, alpha=0.18, zorder=3)
        _stats_box(ax, f"{len(pts)} touches  ·  {n} games", color=color)

    fig.savefig(OUTPUT_DIR / '01_touch_map.png', dpi=160, bbox_inches='tight', facecolor=BG_COLOR)
    plt.close()
    print('  ✓ 01_touch_map.png')

In [8]:
# ═══════════════════════════════════════════════════════════════
# PLOT 2 — PASS MAP
# ═══════════════════════════════════════════════════════════════
#
#  Normal passes       → thin player-colour arrow
#  Final third passes  → thick contrasting arrow (GOLD)
#  Into opposition box → thick contrasting arrow + ★ at the end (cyan)

def plot_pass_map(all_accs, all_n):
    p = _pitch()
    fig, axes = _make_3way_fig(
        title    = f'{PLAYERS[0]["name"]}  vs  {PLAYERS[1]["name"]}  vs  {PLAYERS[2]["name"]}  —  Pass Map',
        subtitle = f'{COMPETITION}  ·  attacking left → right',
        note     = (
            'Thin arrow = normal pass  ·  Thick GOLD arrow = final-third pass  ·  '
            'Thick CYAN arrow + ★ = pass into opposition box'
        ),
    )

    for ax, player_cfg, acc, n in zip(axes, PLAYERS, all_accs, all_n):
        _label(ax, player_cfg)
        color = player_cfg['color']
        succ  = acc['succ_passes']
        fail  = acc['fail_passes']
        box_set  = set(id(ev) for ev in acc['box_passes'])
        ft_set   = set(id(ev) for ev in acc['final_third_passes']) - box_set

        # Failed passes — thin grey
        for ev in fail:
            sx, sy = xy(ev); ex, ey = end_xy(ev)
            if None in (sx, sy, ex, ey): continue
            p.arrows(sx, sy, ex, ey, ax=ax, color=GREY,
                     width=0.7, headwidth=2.0, headlength=2.0, alpha=0.22, zorder=2)

        # Normal successful passes — thin player colour
        for ev in succ:
            if id(ev) in box_set or id(ev) in ft_set: continue
            sx, sy = xy(ev); ex, ey = end_xy(ev)
            if None in (sx, sy, ex, ey): continue
            p.arrows(sx, sy, ex, ey, ax=ax, color=color,
                     width=0.9, headwidth=2.4, headlength=2.4, alpha=0.30, zorder=3)

        # Final-third passes — thick GOLD
        for ev in succ:
            if id(ev) not in ft_set: continue
            sx, sy = xy(ev); ex, ey = end_xy(ev)
            if None in (sx, sy, ex, ey): continue
            p.arrows(sx, sy, ex, ey, ax=ax, color=GOLD,
                     width=2.2, headwidth=5.5, headlength=5.5, alpha=0.85, zorder=4)

        # Box passes — thick CYAN arrow + ★ end marker
        for ev in acc['box_passes']:
            sx, sy = xy(ev); ex, ey = end_xy(ev)
            if None in (sx, sy, ex, ey): continue
            p.arrows(sx, sy, ex, ey, ax=ax, color='#00E5FF',
                     width=2.8, headwidth=6.5, headlength=6.5, alpha=0.92, zorder=5)
            p.scatter(ex, ey, ax=ax, s=280, color='#00E5FF', marker='*',
                      edgecolors=color, linewidths=0.9, alpha=0.97, zorder=6)

        total = len(succ) + len(fail)
        pct   = len(succ) / total * 100 if total else 0
        legend = [
            Line2D([0],[0], marker='>', color=color, lw=0, markersize=9,
                   label=f'Succ ({len(succ)})  |  Fail ({len(fail)})  |  {pct:.0f}% acc'),
            Line2D([0],[0], marker='>', color=GOLD, lw=0, markersize=11,
                   label=f'Into final third ({len(acc["final_third_passes"])} total)'),
            Line2D([0],[0], marker='*', color='#00E5FF', lw=0, markersize=14,
                   markeredgecolor=color, markeredgewidth=0.8,
                   label=f'Into opp. box ({len(acc["box_passes"])})'),
        ]
        ax.legend(handles=legend, loc='lower left', fontsize=16,
                  facecolor=BG_COLOR, edgecolor=color, labelcolor=TEXT_COLOR)
        _stats_box(ax, f"{total} passes  ·  {pct:.0f}% acc  ·  {n} games", color=color)

    fig.savefig(OUTPUT_DIR / '02_pass_map.png', dpi=160, bbox_inches='tight', facecolor=BG_COLOR)
    plt.close()
    print('  ✓ 02_pass_map.png')

In [9]:
# ═══════════════════════════════════════════════════════════════
# PLOT 3 — FAST PASS MAP  (passes within 2 s of reception)
# ═══════════════════════════════════════════════════════════════

def plot_fast_pass_map(all_accs, all_n):
    p = _pitch()
    fig, axes = _make_3way_fig(
        title    = f'{PLAYERS[0]["name"]}  vs  {PLAYERS[1]["name"]}  vs  {PLAYERS[2]["name"]}  —  Fast Pass Map',
        subtitle = f'{COMPETITION}  ·  Passes released within 2 s of receiving  ·  attacking left → right',
        note     = 'Arrow colour = player  ·  Grey = unsuccessful  ·  Gold ★ = into final third  ·  Cyan ◆ = into box',
    )

    for ax, player_cfg, acc, n in zip(axes, PLAYERS, all_accs, all_n):
        _label(ax, player_cfg)
        color = player_cfg['color']
        succ  = acc['fast_succ']
        fail  = acc['fast_fail']
        total = len(succ) + len(fail)
        pct   = len(succ) / total * 100 if total else 0

        for ev in fail:
            sx, sy = xy(ev); ex, ey = end_xy(ev)
            if None in (sx, sy, ex, ey): continue
            p.arrows(sx, sy, ex, ey, ax=ax, color=GREY,
                     width=1.0, headwidth=2.8, headlength=2.8, alpha=0.30, zorder=2)

        ft_count = box_count = 0
        for ev in succ:
            sx, sy = xy(ev); ex, ey = end_xy(ev)
            if None in (sx, sy, ex, ey): continue
            into_box   = (ex > 83.0) and (21.1 <= ey <= 78.9)
            into_final = (ex > 66.67) and not into_box
            p.arrows(sx, sy, ex, ey, ax=ax, color=color,
                     width=1.6, headwidth=4.0, headlength=4.0, alpha=0.65, zorder=3)
            if into_box:
                p.scatter(ex, ey, ax=ax, s=200, color='#00E5FF', marker='D',
                          edgecolors=color, linewidths=0.8, alpha=0.97, zorder=5)
                box_count += 1
            elif into_final:
                p.scatter(ex, ey, ax=ax, s=160, color=GOLD, marker='*',
                          edgecolors=color, linewidths=0.7, alpha=0.95, zorder=4)
                ft_count += 1

        legend = [
            Line2D([0],[0], marker='>', color=color, lw=0, markersize=11,
                   label=f'Succ fast: {len(succ)}'),
            Line2D([0],[0], marker='>', color=GREY,  lw=0, markersize=9,
                   label=f'Unsucc: {len(fail)}'),
            Line2D([0],[0], marker='*', color=GOLD,  lw=0, markersize=15,
                   label=f'Into final third: {ft_count}'),
            Line2D([0],[0], marker='D', color='#00E5FF', lw=0, markersize=16,
                   markeredgecolor=color, markeredgewidth=0.8,
                   label=f'Into opp. box: {box_count}'),
        ]
        ax.legend(handles=legend, loc='lower left', fontsize=16,
                  facecolor=BG_COLOR, edgecolor=color, labelcolor=TEXT_COLOR)
        _stats_box(ax,
                   f"{total} fast passes  ·  {pct:.0f}% acc\n"
                   f"{ft_count} final third  ·  {box_count} into box  ·  {n} games",
                   color=color)

    fig.savefig(OUTPUT_DIR / '03_fast_pass_map.png', dpi=160, bbox_inches='tight', facecolor=BG_COLOR)
    plt.close()
    print('  ✓ 03_fast_pass_map.png')

In [10]:
# ═══════════════════════════════════════════════════════════════
# PLOT 4 — PROGRESSIVE CARRIES
# ═══════════════════════════════════════════════════════════════

def plot_progressive_carries(all_accs, all_n):
    p = _pitch()
    fig, axes = _make_3way_fig(
        title    = f'{PLAYERS[0]["name"]}  vs  {PLAYERS[1]["name"]}  vs  {PLAYERS[2]["name"]}  —  Progressive Carries',
        subtitle = f'{COMPETITION}  ·  ≥10 metres forward  ·  attacking left → right',
        note     = 'Arrow = carry direction  ·  Cyan filled arrowhead = carry ending in opposition box',
    )

    for ax, player_cfg, acc, n in zip(axes, PLAYERS, all_accs, all_n):
        _label(ax, player_cfg)
        color = player_cfg['color']
        prog  = acc['prog_carries']
        box_c = acc['box_carries']
        box_ids = set(id(c) for c in box_c)

        for c in prog:
            into_box = id(c) in box_ids
            arrow_color  = '#00E5FF' if into_box else color
            arrow_alpha  = 0.92 if into_box else 0.65
            arrow_width  = 2.8  if into_box else 1.8
            p.arrows(c['x'], c['y'], c['endX'], c['endY'], ax=ax,
                     color=arrow_color, width=arrow_width,
                     headwidth=arrow_width*2.2, headlength=arrow_width*2.2,
                     alpha=arrow_alpha, zorder=4 if into_box else 3)
            p.scatter(c['x'], c['y'], ax=ax, s=28, color=color,
                      alpha=0.55, zorder=2)
            if into_box:
                # Label the box-entry carry endpoint
                p.scatter(c['endX'], c['endY'], ax=ax, s=260, color='#00E5FF',
                          marker='D', edgecolors=color, linewidths=1.0,
                          alpha=0.97, zorder=6)

        legend = [
            Line2D([0],[0], marker='>', color=color, lw=0, markersize=11,
                   label=f'Progressive carries: {len(prog)}'),
            Line2D([0],[0], marker='D', color='#00E5FF', lw=0, markersize=13,
                   markeredgecolor=color, markeredgewidth=0.9,
                   label=f'Into opp. box: {len(box_c)}'),
        ]
        ax.legend(handles=legend, loc='lower left', fontsize=16,
                  facecolor=BG_COLOR, edgecolor=color, labelcolor=TEXT_COLOR)
        _stats_box(ax, f"{len(prog)} prog. carries  ·  {len(box_c)} into box  ·  {n} games", color=color)

    fig.savefig(OUTPUT_DIR / '04_progressive_carries.png', dpi=160, bbox_inches='tight', facecolor=BG_COLOR)
    plt.close()
    print('  ✓ 04_progressive_carries.png')

In [11]:
# ═══════════════════════════════════════════════════════════════
# PLOT 5 — POST PROGRESSIVE CARRY ACTION
# ═══════════════════════════════════════════════════════════════
#  After a progressive carry, what does the player do?
#  Pass  → arrow from carry endpoint
#  Shot  → filled circle at shot origin (player colour = successful, red = missed)

def plot_post_carry_action(all_accs, all_n):
    p = _pitch()
    fig, axes = _make_3way_fig(
        title    = f'{PLAYERS[0]["name"]}  vs  {PLAYERS[1]["name"]}  vs  {PLAYERS[2]["name"]}  —  Post-Progressive-Carry Action',
        subtitle = f'{COMPETITION}  ·  Action immediately following a progressive carry  ·  attacking left → right',
        note     = 'Arrow = pass after carry  ·  Filled circle = shot (green=on target, orange=off/blocked)  ·  Start = carry end point',
    )

    for ax, player_cfg, acc, n in zip(axes, PLAYERS, all_accs, all_n):
        _label(ax, player_cfg)
        color   = player_cfg['color']
        actions = acc['post_carry_actions']

        n_pass  = n_shot = n_other = 0
        for act in actions:
            sx, sy = act['sx'], act['sy']  # carry endpoint = action start
            if act['is_pass']:
                ex, ey = act['ex'], act['ey']
                if None in (sx, sy, ex, ey): continue
                arrow_col = color if act['is_successful'] else GREY
                p.arrows(sx, sy, ex, ey, ax=ax, color=arrow_col,
                         width=1.6, headwidth=4.2, headlength=4.2, alpha=0.72, zorder=4)
                n_pass += 1
            elif act['is_shot']:
                shot_x = act.get('ex_shot') or sx
                shot_y = act.get('ey_shot') or sy
                # On-target shots (SavedShot=10, Goal=16)
                on_target = act['next_type'] in {10, 16}
                dot_color = '#00C853' if on_target else '#FF6D00'
                p.scatter(shot_x, shot_y, ax=ax, s=260, color=dot_color,
                          edgecolors=color, linewidths=1.2, alpha=0.95, zorder=6)
                p.scatter(sx, sy, ax=ax, s=60, color=color, alpha=0.55, zorder=5)
                n_shot += 1
            else:
                n_other += 1

        legend = [
            Line2D([0],[0], marker='>', color=color, lw=0, markersize=13,
                   label=f'Pass after carry: {n_pass}'),
            Line2D([0],[0], marker='>', color=GREY,  lw=0, markersize=9,
                   label='Pass (unsuccessful)'),
            Line2D([0],[0], marker='o', color='#00C853', lw=0, markersize=14,
                   markeredgecolor=color, label=f'Shot on target: {n_shot}'),
            Line2D([0],[0], marker='o', color='#FF6D00', lw=0, markersize=12,
                   markeredgecolor=color, label='Shot off/blocked'),
        ]
        ax.legend(handles=legend, loc='lower left', fontsize=16,
                  facecolor=BG_COLOR, edgecolor=color, labelcolor=TEXT_COLOR)
        _stats_box(ax,
                   f"{len(actions)} carries tracked\n"
                   f"Pass: {n_pass}  ·  Shot: {n_shot}  ·  Other: {n_other}\n{n} games",
                   color=color)

    fig.savefig(OUTPUT_DIR / '05_post_carry_action.png', dpi=160, bbox_inches='tight', facecolor=BG_COLOR)
    plt.close()
    print('  ✓ 05_post_carry_action.png')

In [12]:
# ═══════════════════════════════════════════════════════════════
# PLOT 6 — BALL RECOVERY MAP
# ═══════════════════════════════════════════════════════════════
#  Tackles ★  |  Interceptions ▲  |  Ball Recoveries ●
#  All marks are size-differentiated and colour-coded for contrast

def plot_ball_recovery_map(all_accs, all_n):
    p = _pitch()
    fig, axes = _make_3way_fig(
        title    = f'{PLAYERS[0]["name"]}  vs  {PLAYERS[1]["name"]}  vs  {PLAYERS[2]["name"]}  —  Ball Recovery Map',
        subtitle = f'{COMPETITION}  ·  attacking left → right',
        note     = 'Gold ★ = Tackle Won  ·  Red ✕ = Tackle Lost  ·  Cyan ▲ = Interception  ·  Player colour ● = Ball Recovery',
    )

    for ax, player_cfg, acc, n in zip(axes, PLAYERS, all_accs, all_n):
        _label(ax, player_cfg)
        color   = player_cfg['color']
        all_def = acc['tack_won'] + acc['tack_lost'] + acc['interceptions'] + acc['loose_rec']
        xs_all  = _safe_xs(all_def)
        ys_all  = _safe_ys(all_def)

        if len(xs_all) >= 5:
            p.kdeplot(xs_all, ys_all, ax=ax, fill=True, levels=45,
                      cmap='RdPu', alpha=0.28, zorder=2, bw_adjust=1.25)

        STYLES = [
            (acc['tack_won'],     GOLD,       '*', 240, 0.95, f'Tackle Won ({len(acc["tack_won"])})'),
            (acc['tack_lost'],    '#E74C3C',  'X', 140, 0.85, f'Tackle Lost ({len(acc["tack_lost"])})'),
            (acc['interceptions'],'#00E5FF',  '^', 170, 0.90, f'Interception ({len(acc["interceptions"])})'),
            (acc['loose_rec'],    color,      'o', 120, 0.80, f'Recovery ({len(acc["loose_rec"])})'),
        ]
        handles = []
        for evs, c, mk, sz, al, lbl in STYLES:
            if not evs: continue
            xs = _safe_xs(evs); ys = _safe_ys(evs)
            if not xs: continue
            p.scatter(xs, ys, ax=ax, s=sz, color=c, marker=mk,
                      edgecolors='white', linewidths=0.7, alpha=al, zorder=4)
            handles.append(Line2D([0],[0], marker=mk, color=c, lw=0,
                                  markersize=14, label=lbl))

        ax.legend(handles=handles, loc='lower left', fontsize=16,
                  facecolor=BG_COLOR, edgecolor=color, labelcolor=TEXT_COLOR)

        won   = len(acc['tack_won']) + len(acc['interceptions']) + len(acc['loose_rec'])
        total = won + len(acc['tack_lost'])
        _stats_box(ax, f"{total} actions  ·  {won} balls won  ·  {n} games", color=color)

    fig.savefig(OUTPUT_DIR / '06_ball_recovery_map.png', dpi=160, bbox_inches='tight', facecolor=BG_COLOR)
    plt.close()
    print('  ✓ 06_ball_recovery_map.png')

In [13]:
# ═══════════════════════════════════════════════════════════════
# PLOT 7 — POST-RECOVERY PASS MAP  (within 3 s of winning ball)
# ═══════════════════════════════════════════════════════════════

def plot_post_recovery_pass_map(all_accs, all_n):
    p = _pitch()
    fig, axes = _make_3way_fig(
        title    = f'{PLAYERS[0]["name"]}  vs  {PLAYERS[1]["name"]}  vs  {PLAYERS[2]["name"]}  —  Post-Recovery Pass Map',
        subtitle = f'{COMPETITION}  ·  Pass within 3 s of tackle won / interception / ball recovery  ·  attacking left → right',
        note     = 'Player arrow = successful  ·  Grey = unsuccessful  ·  Gold ★ = into final third  ·  Cyan ◆ = into box',
    )

    for ax, player_cfg, acc, n in zip(axes, PLAYERS, all_accs, all_n):
        _label(ax, player_cfg)
        color = player_cfg['color']
        succ  = acc['post_rec_succ']
        fail  = acc['post_rec_fail']
        total = len(succ) + len(fail)
        pct   = len(succ) / total * 100 if total else 0

        all_post = succ + fail
        if len(all_post) >= 5:
            p.kdeplot(_safe_xs(all_post), _safe_ys(all_post), ax=ax,
                      fill=True, levels=50, cmap='YlOrRd', alpha=0.22,
                      zorder=2, bw_adjust=1.10)

        for ev in fail:
            sx, sy = xy(ev); ex, ey = end_xy(ev)
            if None in (sx, sy, ex, ey): continue
            p.arrows(sx, sy, ex, ey, ax=ax, color=GREY,
                     width=1.0, headwidth=2.8, headlength=2.8, alpha=0.35, zorder=3)

        ft_count = box_count = 0
        for ev in succ:
            sx, sy = xy(ev); ex, ey = end_xy(ev)
            if None in (sx, sy, ex, ey): continue
            into_box   = (ex > 83.0) and (21.1 <= ey <= 78.9)
            into_final = (ex > 66.67) and not into_box
            p.arrows(sx, sy, ex, ey, ax=ax, color=color,
                     width=1.6, headwidth=4.0, headlength=4.0, alpha=0.65, zorder=4)
            if into_box:
                p.scatter(ex, ey, ax=ax, s=200, color='#00E5FF', marker='D',
                          edgecolors=color, linewidths=0.8, alpha=0.97, zorder=6)
                box_count += 1
            elif into_final:
                p.scatter(ex, ey, ax=ax, s=160, color=GOLD, marker='*',
                          edgecolors=color, linewidths=0.7, alpha=0.95, zorder=5)
                ft_count += 1

        legend = [
            Line2D([0],[0], marker='>', color=color, lw=0, markersize=11,
                   label=f'Succ: {len(succ)}  ·  Unsucc: {len(fail)}'),
            Line2D([0],[0], marker='*', color=GOLD,  lw=0, markersize=14,
                   label=f'Into final third: {ft_count}'),
            Line2D([0],[0], marker='D', color='#00E5FF', lw=0, markersize=14,
                   markeredgecolor=color, markeredgewidth=0.8,
                   label=f'Into opp. box: {box_count}'),
        ]
        ax.legend(handles=legend, loc='lower left', fontsize=16,
                  facecolor=BG_COLOR, edgecolor=color, labelcolor=TEXT_COLOR)
        _stats_box(ax,
                   f"{total} post-rec passes  ·  {pct:.0f}% acc\n"
                   f"{ft_count} final third  ·  {box_count} into box  ·  {n} games",
                   color=color)

    fig.savefig(OUTPUT_DIR / '07_post_recovery_pass_map.png', dpi=160, bbox_inches='tight', facecolor=BG_COLOR)
    plt.close()
    print('  ✓ 07_post_recovery_pass_map.png')

In [14]:
# ═══════════════════════════════════════════════════════════════
# PLOT 8 — xT VIA PASSES PER ZONE  (all three players, one visual)
# ═══════════════════════════════════════════════════════════════
#
# Layout: 1 row of 3 mini-pitches side-by-side, each showing NET xT
# per origin zone for one player.  A shared colourbar spans all three.
# Coordinate notes:
#   • XT_GRID rows  = y-axis bins  (row 0 = bottom of pitch, row 11 = top)
#   • XT_GRID cols  = x-axis bins  (col 0 = own goal, col 15 = opp goal)
#   • Cell centres cx[col], cy[row] → Opta x,y units → drawn correctly
#     by mplsoccer's heatmap with x_grid / y_grid edges.

def plot_xt_three_way(all_accs, all_n):
    # Pre-compute per player
    player_data = []
    global_max_abs = 1e-6
    for acc, n, pcfg in zip(all_accs, all_n, PLAYERS):
        pxt   = compute_pass_xt_full(acc['succ_passes'])
        znet  = compute_xt_zones_net(pxt)
        gross = sum(d['xt_end']    for d in pxt)
        net   = sum(d['xt_gained'] for d in pxt)
        top20 = sorted([d for d in pxt if d['xt_gained'] > 0],
                       key=lambda d: d['xt_gained'], reverse=True)[:20]
        player_data.append({
            'cfg': pcfg, 'n': n,
            'pxt': pxt, 'znet': znet,
            'gross': gross, 'net': net, 'top20': top20,
        })
        global_max_abs = max(global_max_abs, float(np.abs(znet).max()))

    # Grid edges in Opta units (correct for mplsoccer pitch_type='opta')
    x_grid = np.linspace(0, 100, N_COLS + 1)  # 17 edges → 16 cols
    y_grid = np.linspace(0, 100, N_ROWS + 1)  # 13 edges → 12 rows
    cx     = (x_grid[:-1] + x_grid[1:]) / 2.0  # cell centre x
    cy     = (y_grid[:-1] + y_grid[1:]) / 2.0  # cell centre y

    # Figure: 2-on-top + 1-centred-below  (matches _make_3way_fig layout)
    from matplotlib.gridspec import GridSpec
    fig = plt.figure(figsize=(36, 34))
    fig.set_facecolor(BG_COLOR)

    gs  = GridSpec(2, 4, figure=fig,
                   hspace=0.18, wspace=0.08,
                   top=0.85, bottom=0.08, left=0.02, right=0.93)
    ax0 = fig.add_subplot(gs[0, :2])   # top-left
    ax1 = fig.add_subplot(gs[0, 2:])   # top-right
    ax2 = fig.add_subplot(gs[1, 1:3])  # bottom, centred
    axes = [ax0, ax1, ax2]

    p = _pitch()
    for ax in axes:
        p.draw(ax=ax)
        ax.set_facecolor(BG_COLOR)

    # Shared colourmap  (diverging: red = net negative, colour = net positive)
    all_pct_vals = np.concatenate([pd['znet'][pd['znet'] != 0].ravel() for pd in player_data])
    if len(all_pct_vals) > 5:
        clim = float(np.percentile(np.abs(all_pct_vals), 95))
        global_max_abs = clim

    # One shared diverging colormap for ALL three players, so the single
    # colourbar on the right is valid for every subplot (red = net negative,
    # blue = net positive — independent of each player's own team colour).
    SHARED_CMAP = LinearSegmentedColormap.from_list(
        'xt_net_shared', ['#BF360C', BG_COLOR, '#1565C0'], N=256
    )

    pcm_handles = []
    for ax, pd_ in zip(axes, player_data):
        pcfg   = pd_['cfg']
        color  = pcfg['color']
        znet   = pd_['znet']

        stats = dict(statistic=znet, x_grid=x_grid, y_grid=y_grid, cx=cx, cy=cy)
        pcm   = p.heatmap(stats, ax=ax, cmap=SHARED_CMAP,
                           vmin=-global_max_abs, vmax=global_max_abs,
                           alpha=0.88, zorder=2)
        pcm_handles.append((pcm, color, pcfg['name']))

        # Grid lines (xT cell boundaries)
        for col in range(N_COLS + 1):
            ax.axvline(col * CELL_W, color=PITCH_LINE, alpha=0.15, lw=0.5, zorder=3)
        for row in range(N_ROWS + 1):
            ax.axhline(row * CELL_H, color=PITCH_LINE, alpha=0.15, lw=0.5, zorder=3)

        # Zone value labels (only non-zero cells, min font to avoid clutter)
        threshold = global_max_abs * 0.05
        for row in range(N_ROWS):
            for col in range(N_COLS):
                val = znet[row, col]
                if abs(val) < threshold:
                    continue
                brightness = abs(val) / (global_max_abs + 1e-9)
                txt_c = 'white' if brightness > 0.55 else TEXT_COLOR
                # cy[row] is the Opta y-coordinate for cell row
                ax.text(cx[col], cy[row], f'{val:+.3f}',
                        ha='center', va='center', fontsize=6.5, color=txt_c,
                        fontfamily='monospace', alpha=0.93, zorder=5)

        # Top-20 passes by net xT gain
        for d in pd_['top20']:
            p.arrows(d['x'], d['y'], d['endX'], d['endY'], ax=ax,
                     color='#1A1A1A', width=3.2, headwidth=7.0,
                     headlength=7.0, alpha=0.65, zorder=6)
            p.arrows(d['x'], d['y'], d['endX'], d['endY'], ax=ax,
                     color=GOLD, width=1.8, headwidth=4.5,
                     headlength=4.5, alpha=0.97, zorder=7)

        # Per-player sub-title
        ax.set_title(
            f"{pcfg['name']}  ·  {pcfg['team']}\n"
            f"Gross xT: {pd_['gross']:.3f}   Net xT: {pd_['net']:+.3f}   "
            f"{len(pd_['pxt'])} succ. passes   {pd_['n']} games",
            fontsize=24, fontweight='bold', color=color, pad=9,
        )

        legend = [
            Line2D([0],[0], marker='>', color=GOLD, lw=0, markersize=16,
                   markeredgecolor='#1A1A1A', markeredgewidth=0.8,
                   label=f'Top-20 passes by net xT'),
        ]
        ax.legend(handles=legend, loc='lower left', fontsize=10,
                  facecolor=BG_COLOR, edgecolor=color, labelcolor=TEXT_COLOR)

    # Shared colourbar on the right (same SHARED_CMAP used by every subplot)
    cbar_ax = fig.add_axes([0.94, 0.10, 0.012, 0.70])
    norm   = Normalize(vmin=-global_max_abs, vmax=global_max_abs)
    sm     = ScalarMappable(cmap=SHARED_CMAP, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, cax=cbar_ax)
    cbar.set_label('Net xT per zone  [ – = threat lost ]',
                   fontsize=18, color=TEXT_COLOR, labelpad=11)
    cbar.ax.tick_params(labelcolor=TEXT_COLOR, labelsize=11)
    cbar.outline.set_edgecolor(PITCH_LINE)

    # Overall title
    fig.text(0.47, 0.98,
             f'{PLAYERS[0]["name"]}  vs  {PLAYERS[1]["name"]}  vs  {PLAYERS[2]["name"]}  —  '
             'xT Created via Passes per Zone',
             ha='center', va='top', fontsize=36, fontweight='bold', color=TEXT_COLOR)
    fig.text(0.47, 0.93,
             f'{COMPETITION}  ·  Karun Singh xT model (16×12 grid)  ·  '
             'NET xT = Σ [xT(dest) − xT(origin)]  ·  attacking left → right',
             ha='center', va='top', fontsize=26, color=TEXT_COLOR, alpha=0.90)
    fig.text(0.47, 0.88, CREDIT_LINE,
             ha='center', va='top', fontsize=20, color=TEXT_COLOR, alpha=0.85, style='italic')

    fig.savefig(OUTPUT_DIR / '08_xt_zone_comparison.png',
                dpi=150, bbox_inches='tight', facecolor=BG_COLOR)
    plt.close()
    print('  ✓ 08_xt_zone_comparison.png')

In [15]:
# ═══════════════════════════════════════════════════════════════
# PLOTS 9 + 10 — PRE-RECEIVE & PRE-DEFENSIVE DISTANCE
# ═══════════════════════════════════════════════════════════════
#
# Both plots are bar charts (not pitch maps) comparing the three
# players on average distance covered in the relevant context.
# They are rendered as a 2-panel figure side by side.

def plot_distance_charts(all_accs, all_n):
    fig, axes = plt.subplots(1, 2, figsize=(24, 10))
    fig.set_facecolor(BG_COLOR)
    fig.subplots_adjust(wspace=0.22, top=0.82, bottom=0.10,
                        left=0.07, right=0.97)

    names   = [p['name'] for p in PLAYERS]
    colors  = [p['color'] for p in PLAYERS]

    recv_avgs  = []
    recv_sds   = []
    def_avgs   = []
    def_sds    = []
    recv_ns    = []
    def_ns     = []

    for acc in all_accs:
        rd = acc['pre_recv_dists']
        dd = acc['pre_def_dists']
        recv_avgs.append(np.mean(rd) if rd else 0)
        recv_sds.append( np.std(rd)  if rd else 0)
        def_avgs.append( np.mean(dd) if dd else 0)
        def_sds.append(  np.std(dd)  if dd else 0)
        recv_ns.append(len(rd))
        def_ns.append(len(dd))

    x = np.arange(len(PLAYERS))

    # ── Panel 1: Pre-receive distance ──
    ax1 = axes[0]
    ax1.set_facecolor(BG_COLOR)
    bars = ax1.bar(x, recv_avgs, color=colors, alpha=0.82,
                   edgecolor='white', linewidth=1.2, width=0.55)
    ax1.errorbar(x, recv_avgs, yerr=recv_sds, fmt='none',
                 ecolor=TEXT_COLOR, elinewidth=1.5, capsize=6, alpha=0.70)
    ax1.set_xticks(x)
    ax1.set_xticklabels(
        [f"{PLAYERS[i]['name']}\n{PLAYERS[i]['team']})" for i in range(len(PLAYERS))],
        fontsize=14, color=TEXT_COLOR
    )
    ax1.set_ylabel('Avg distance (metres)', fontsize=13, color=TEXT_COLOR)
    ax1.set_title('Avg Distance Covered\nBefore Receiving the Ball',
                  fontsize=18, fontweight='bold', color=TEXT_COLOR, pad=10)
    ax1.tick_params(axis='y', labelcolor=TEXT_COLOR)
    ax1.spines[['top','right']].set_visible(False)
    ax1.set_facecolor(BG_COLOR)
    for bar, val in zip(bars, recv_avgs):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                 f'{val:.2f}', ha='center', va='bottom',
                 fontsize=14, fontweight='bold', color=TEXT_COLOR)

    # ── Panel 2: Pre-defensive distance ──
    ax2 = axes[1]
    ax2.set_facecolor(BG_COLOR)
    bars2 = ax2.bar(x, def_avgs, color=colors, alpha=0.82,
                    edgecolor='white', linewidth=1.2, width=0.55)
    ax2.errorbar(x, def_avgs, yerr=def_sds, fmt='none',
                 ecolor=TEXT_COLOR, elinewidth=1.5, capsize=6, alpha=0.70)
    ax2.set_xticks(x)
    ax2.set_xticklabels(
        [f"{PLAYERS[i]['name']}\n{PLAYERS[i]['team']})" for i in range(len(PLAYERS))],
        fontsize=14, color=TEXT_COLOR
    )
    ax2.set_ylabel('Avg distance (metres)', fontsize=13, color=TEXT_COLOR)
    ax2.set_title('Avg Distance Covered\nBefore a Defensive Action',
                  fontsize=18, fontweight='bold', color=TEXT_COLOR, pad=10)
    ax2.tick_params(axis='y', labelcolor=TEXT_COLOR)
    ax2.spines[['top','right']].set_visible(False)
    for bar, val in zip(bars2, def_avgs):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                 f'{val:.2f}', ha='center', va='bottom',
                 fontsize=14, fontweight='bold', color=TEXT_COLOR)

    fig.text(0.5, 1.07,
             f'{PLAYERS[0]["name"]}  vs  {PLAYERS[1]["name"]}  vs  {PLAYERS[2]["name"]}  —  Distance Metrics',
             ha='center', va='top', fontsize=28, fontweight='bold', color=TEXT_COLOR)
    fig.text(0.5, 1.02,
             f'{COMPETITION}  ·  Error bars = ±1 SD  ·  n = displacement samples from Opta events',
             ha='center', va='top', fontsize=18, color=TEXT_COLOR, alpha=0.85)
    fig.text(0.5, 0.97, CREDIT_LINE,
             ha='center', va='top', fontsize=15, color=TEXT_COLOR, alpha=0.80, style='italic')

    fig.savefig(OUTPUT_DIR / '09_10_distance_metrics.png',
                dpi=160, bbox_inches='tight', facecolor=BG_COLOR)
    plt.close()
    print('  ✓ 09_10_distance_metrics.png')

In [16]:
# ═══════════════════════════════════════════════════════════════
# MAIN  —  run this cell to scrape + render all visuals
# ═══════════════════════════════════════════════════════════════

def main():
    # Guard: ensure match IDs are real (not just placeholders)
    placeholder_warn = []
    for _p in PLAYERS:
        if not _p['match_ids']:
            placeholder_warn.append(_p['name'])
    if placeholder_warn:
        print('[!] Missing match IDs for:', placeholder_warn)
        print('    Fill in match_suffixes in the CONFIG cell, then re-run.')
        return

    print('=' * 72)
    print(f"  {PLAYERS[0]['name']}  vs  {PLAYERS[1]['name']}  vs  {PLAYERS[2]['name']}")
    print(f'  Competition : {COMPETITION}')
    print('=' * 72)
    for _p in PLAYERS:
        print(f"  {_p['name']:<16} id={_p['player_id']}  "
              f"matches={len(_p['match_ids'])}  "
              f"sample_id={_p['match_ids'][0]}")
    print()

    # 1. SCRAPE ──────────────────────────────────────────────────
    print('[1/3] Scraping WhoScored...')
    driver = get_driver()
    all_accs, all_n = [], []
    for player_cfg in PLAYERS:
        print(f'\n  ── {player_cfg["name"]} ({player_cfg["team"]}) ──────────────────────')
        acc, n = scrape_player(driver, player_cfg)
        all_accs.append(acc)
        all_n.append(n)
    driver.quit()

    if all(n == 0 for n in all_n):
        print('\n[!] No data collected for any player. Verify match IDs.')
        return

    # 2. SUMMARY ─────────────────────────────────────────────────
    print('\n[2/3] Summary:')
    for player_cfg, acc, n in zip(PLAYERS, all_accs, all_n):
        total_pass = len(acc['succ_passes']) + len(acc['fail_passes'])
        pass_acc   = len(acc['succ_passes']) / total_pass * 100 if total_pass else 0
        fast_total = len(acc['fast_succ']) + len(acc['fast_fail'])
        rec_total  = len(acc['post_rec_succ']) + len(acc['post_rec_fail'])
        pxt_data   = compute_pass_xt_full(acc['succ_passes'])
        gross_xt   = sum(d['xt_end']    for d in pxt_data)
        net_xt     = sum(d['xt_gained'] for d in pxt_data)
        recv_avg   = (sum(acc['pre_recv_dists']) / len(acc['pre_recv_dists'])
                      if acc['pre_recv_dists'] else 0)
        def_avg    = (sum(acc['pre_def_dists'])  / len(acc['pre_def_dists'])
                      if acc['pre_def_dists'] else 0)

        print(f"\n  ── {player_cfg['name']}  ({n} games) ──────────────────────────")
        print(f"    Passes          : {total_pass}  ({pass_acc:.0f}% acc)")
        print(f"    Final-3rd       : {len(acc['final_third_passes'])}  (box: {len(acc['box_passes'])})")
        print(f"    Fast passes ≤2s : {fast_total}")
        print(f"    Post-rec passes : {rec_total}")
        print(f"    Prog. carries   : {len(acc['prog_carries'])}  (into box: {len(acc['box_carries'])})")
        print(f"    Post-carry acts : {len(acc['post_carry_actions'])}")
        print(f"    Tackles won/lost: {len(acc['tack_won'])} / {len(acc['tack_lost'])}")
        print(f"    Interceptions   : {len(acc['interceptions'])}")
        print(f"    Ball recoveries : {len(acc['loose_rec'])}")
        print(f"    Gross xT        : {gross_xt:.3f}")
        print(f"    Net   xT        : {net_xt:+.3f}")
        print(f"    Pre-recv dist   : {recv_avg:.2f} Opta units (n={len(acc['pre_recv_dists'])})")
        print(f"    Pre-def dist    : {def_avg:.2f} Opta units (n={len(acc['pre_def_dists'])})")

    # 3. RENDER ──────────────────────────────────────────────────
    print('\n[3/3] Rendering visuals...')
    plot_touch_map(all_accs, all_n)
    plot_pass_map(all_accs, all_n)
    plot_fast_pass_map(all_accs, all_n)
    plot_progressive_carries(all_accs, all_n)
    plot_post_carry_action(all_accs, all_n)
    plot_ball_recovery_map(all_accs, all_n)
    plot_post_recovery_pass_map(all_accs, all_n)
    plot_xt_three_way(all_accs, all_n)
    plot_distance_charts(all_accs, all_n)

    outputs = [
        '01_touch_map.png',
        '02_pass_map.png',
        '03_fast_pass_map.png',
        '04_progressive_carries.png',
        '05_post_carry_action.png',
        '06_ball_recovery_map.png',
        '07_post_recovery_pass_map.png',
        '08_xt_zone_comparison.png',
        '09_10_distance_metrics.png',
    ]
    print(f'\n✓ All done  —  {len(outputs)} visuals saved to: {OUTPUT_DIR.resolve()}')
    for f in outputs:
        print(f'    {f}')


# ── Entry point ──────────────────────────────────────────────
main()

  A. Bouaddi  vs  L. Camara  vs  M. Sangaré
  Competition : 2025/26 Ligue 1
  A. Bouaddi       id=504830  matches=28  sample_id=1911281
  L. Camara        id=447689  matches=24  sample_id=1911290
  M. Sangaré       id=442971  matches=29  sample_id=1911276

[1/3] Scraping WhoScored...

  ── A. Bouaddi (Lille) ──────────────────────
  [WS] Fetching match 1911281 ...
       ✓ Ayyoub Bouaddi — 25 raw events
  [WS] Fetching match 1911279 ...
       ✓ Ayyoub Bouaddi — 62 raw events
  [WS] Fetching match 1911310 ...
       ✓ Ayyoub Bouaddi — 59 raw events
  [WS] Fetching match 1911336 ...
       ✓ Ayyoub Bouaddi — 57 raw events
  [WS] Fetching match 1911360 ...
       ✓ Ayyoub Bouaddi — 56 raw events
  [WS] Fetching match 1911374 ...
       ✓ Ayyoub Bouaddi — 49 raw events
  [WS] Fetching match 1911307 ...
       ✓ Ayyoub Bouaddi — 53 raw events
  [WS] Fetching match 1911335 ...
       ✓ Ayyoub Bouaddi — 59 raw events
  [WS] Fetching match 1911308 ...
       ✓ Ayyoub Bouaddi — 45 raw events
 

  ✓ 01_touch_map.png
  ✓ 02_pass_map.png
  ✓ 03_fast_pass_map.png
  ✓ 04_progressive_carries.png
  ✓ 05_post_carry_action.png
  ✓ 06_ball_recovery_map.png
  ✓ 07_post_recovery_pass_map.png
  ✓ 08_xt_zone_comparison.png
  ✓ 09_10_distance_metrics.png

✓ All done  —  9 visuals saved to: C:\Users\mizga\Desktop\bouaddi_camara_sangare
    01_touch_map.png
    02_pass_map.png
    03_fast_pass_map.png
    04_progressive_carries.png
    05_post_carry_action.png
    06_ball_recovery_map.png
    07_post_recovery_pass_map.png
    08_xt_zone_comparison.png
    09_10_distance_metrics.png
